# Training of the LoRA-N2V

In [1]:
import pandas as pd
import numpy as np
import math
import torch
import matplotlib.pyplot as plt
from peft import LoraConfig, get_peft_model
from torch import nn
from torch.utils.data.sampler import Sampler
from collections import defaultdict
import random
from tqdm import tqdm
from astropy.io import fits
from photutils import CircularAnnulus, EllipticalAperture
from astropy.stats import sigma_clip
from photutils.aperture import aperture_photometry

/tmp/ipykernel_672/1840529391.py:13: DeprecationWarning: `photutils.CircularAnnulus` is a deprecated alias for `photutils.aperture.CircularAnnulus` and will be removed in the future. Instead, please use `from photutils.aperture import CircularAnnulus` to silence this warning.
  from photutils import CircularAnnulus, EllipticalAperture
/tmp/ipykernel_672/1840529391.py:13: DeprecationWarning: `photutils.EllipticalAperture` is a deprecated alias for `photutils.aperture.EllipticalAperture` and will be removed in the future. Instead, please use `from photutils.aperture import EllipticalAperture` to silence this warning.
  from photutils import CircularAnnulus, EllipticalAperture


# Define the PairedBatchSampler for the Paired difference loss, and other functions used to generate the training inputs. 

In [2]:
class PairedBatchSampler(Sampler):
    """
    choose P (ref_id, filter)
    
    P = 8, K = 2, batch_size = 16
    batch_indices = [idx_A1, idx_G1, ... idx_F1,   idx_A2, idx_G2, ... idx_F2]
    
    The same ref_id AND filter
    """
    def __init__(self, ref_ids, filters, P):
        
        super(PairedBatchSampler, self).__init__()
        
        if P <= 0:
            raise ValueError("P must be > 0")
        
        self.P = P
        self.K_is_fixed_at = 2
        self.batch_size = P * self.K_is_fixed_at
        
        print("constructing PairedBatchSampler ...")
        
        grouped_indices = defaultdict(list)
        for i, (ref_id, filter_val) in enumerate(zip(ref_ids, filters)):
            grouped_indices[(ref_id, filter_val)].append(i)
        
        print(f"Found {len(grouped_indices)} unique (ref_id, filter) combinations")
        
        print(f"creating 'chunks' (size K={self.K_is_fixed_at})...")
        self.all_chunks = []
        for combo, indices in grouped_indices.items():
            if len(indices) >= self.K_is_fixed_at:
                random.shuffle(indices)
                
                num_chunks_for_this_combo = len(indices) // self.K_is_fixed_at
                
                for i in range(num_chunks_for_this_combo):
                    chunk = indices[i * self.K_is_fixed_at : (i + 1) * self.K_is_fixed_at]
                    self.all_chunks.append({
                        'indices': chunk,
                        'ref_id': combo[0],
                        'filter': combo[1]
                    })
        
        print(f"Already created {len(self.all_chunks)} 'K-chunks'.")
        
        if len(self.all_chunks) < P:
            raise ValueError(
                f"Only {len(self.all_chunks)} chunks available, but need P={P}. "
                f"Try using a smaller P or increasing the data."
            )
        
        self.num_batches = len(self.all_chunks) // P
    
    def __iter__(self):
        random.shuffle(self.all_chunks)
        
        for i in range(self.num_batches):
            batch_part1_indices = []
            batch_part2_indices = []
            
            p_chunks = self.all_chunks[i * self.P : (i + 1) * self.P]
            
            for chunk_info in p_chunks:
                chunk = chunk_info['indices']
                batch_part1_indices.append(chunk[0])
                batch_part2_indices.append(chunk[1])
            
            final_batch_indices = batch_part1_indices + batch_part2_indices
            yield final_batch_indices
    
    def __len__(self):
        return self.num_batches

In [4]:
#write the function to generate the mask_annulus, mask_gal
#test the code in the following
def annulus_mask_generator(mask, aperture_x, aperture_y, r_in=30, r_out=45):

    image_shape = (cutout_size*2,cutout_size*2)
    center = (aperture_x, aperture_y)
    annulus_apertures = CircularAnnulus(center, r_in=r_in, r_out=r_out)
    
    mask_object = annulus_apertures.to_mask(method='center')
    mask_annulus = mask_object.to_image(shape=image_shape)
    
    xmask = mask != 0
    
    mask_annulus = mask_annulus * (1 - xmask)
    
    return mask_annulus, annulus_apertures.area

def gal_mask_generator(mask, aperture_x, aperture_y, aperture_theta, aperture_a, aperture_b):

    image_shape = (cutout_size*2,cutout_size*2)
    PIXEL_SCALE = 0.263
    theta = -aperture_theta * np.pi / 180.
    a = aperture_a / PIXEL_SCALE
    b = aperture_b / PIXEL_SCALE

    center = (aperture_x, aperture_y)
    source_aperture = EllipticalAperture(center, a, b, theta)
    mask_object = source_aperture.to_mask(method='exact')
    mask_image_photutils_fractional = mask_object.to_image(shape=image_shape)
    
    xmask = mask != 0
    mask_gal = mask_image_photutils_fractional * (1 - xmask)
    
    return mask_gal#, source_aperture.area

In [3]:
#generate image
cutout_size = 48

def background_annulus(data, mask, aperture_x, aperture_y, r_in=30, r_out=45):
    """Measure background in an annulus."""
    
    masked_data = np.ma.array(data=data, mask=mask != 0)
    masked_data = masked_data.filled(fill_value=0)

    center = (aperture_x, aperture_y)
    annulus_apertures = CircularAnnulus(center, r_in=r_in, r_out=r_out)
    masks = annulus_apertures.to_mask(method='center')

    cutout_data = masks.cutout(masked_data)

    clip_annulus_array = sigma_clip(cutout_data[cutout_data != 0], sigma=3, maxiters=2)

    S = pd.Series()
    S['annulus_mean'] = np.ma.mean(clip_annulus_array)
    S['annulus_median'] = np.ma.median(clip_annulus_array)
    S['annulus_std'] = np.ma.std(clip_annulus_array)
    S['annulus_samples'] = np.ma.count(clip_annulus_array)

    return S

def flux_elliptical(image, mask, aperture_x, aperture_y, aperture_theta, aperture_a, aperture_b):
    """Measure the flux withing an elliptical aperture."""
    
    PIXEL_SCALE = 0.263
    theta = -aperture_theta * np.pi / 180.
    a = aperture_a / PIXEL_SCALE
    b = aperture_b / PIXEL_SCALE

    center = (aperture_x, aperture_y)
    source_aperture = EllipticalAperture(center, a, b, theta)

    xmask = mask != 0
    raw_flux = aperture_photometry(image, source_aperture, mask=xmask)
   
    S = pd.Series()
    S['raw_flux'] = float(raw_flux['aperture_sum'][0])
    S['area'] = source_aperture.area
    
    return S

def creat_stamps(image, sources):
    y = math.floor(sources['aperture_x'].values[0])
    x = math.floor(sources['aperture_y'].values[0])
    #y = math.floor(sources['aperture_x'])
    #x = math.floor(sources['aperture_y'])
    x_start = max((x - cutout_size), 0)
    x_end = min((x + cutout_size), image.shape[0])
    y_start = max((y - cutout_size), 0)
    y_end = min((y + cutout_size), image.shape[1])

    stamps = image[x_start:x_end, y_start:y_end]
    return stamps

def photometry_oneimage(image, mask, aperture_x, aperture_y, aperture_theta, aperture_a, aperture_b):
    
    S1 = background_annulus(image, mask, aperture_x, aperture_y)
    S2 = flux_elliptical(image, mask, aperture_x, aperture_y, aperture_theta, aperture_a, aperture_b)

    flux_obs = S2['raw_flux'] - S2['area'] * S1['annulus_mean']
    return flux_obs, S1['annulus_std']

def generate_mask(size_data, image):

    num_sample = int(size_data[0] * size_data[1] * (1 - ratio))
    mask = np.ones(size_data)
    output = image

    for ich in range(size_data[2]):
        idy_msk = np.random.randint(0, size_data[0], num_sample)
        idx_msk = np.random.randint(0, size_data[1], num_sample)

        idy_neigh = np.random.randint(-size_window[0] // 2 + size_window[0] % 2, size_window[0] // 2 + size_window[0] % 2, num_sample)
        idx_neigh = np.random.randint(-size_window[1] // 2 + size_window[1] % 2, size_window[1] // 2 + size_window[1] % 2, num_sample)

        idy_msk_neigh = idy_msk + idy_neigh
        idx_msk_neigh = idx_msk + idx_neigh

        idy_msk_neigh = idy_msk_neigh + (idy_msk_neigh < 0) * size_data[0] - (idy_msk_neigh >= size_data[0]) * size_data[0]
        idx_msk_neigh = idx_msk_neigh + (idx_msk_neigh < 0) * size_data[1] - (idx_msk_neigh >= size_data[1]) * size_data[1]

        id_msk = (idy_msk, idx_msk, ich)
        id_msk_neigh = (idy_msk_neigh, idx_msk_neigh, ich)

        output[id_msk] = image[id_msk_neigh]
        mask[id_msk] = 0.0

    return output, mask

# Read the training galaxy, and rerange them, generate the training set

In [11]:
path = '../data saved/'

trainingset = pd.read_csv(path + 'selectmediandata.csv')

In [13]:
#Select and arrange the dataframe
P = 8
sampler = PairedBatchSampler(trainingset['ref_id'], trainingset['filter'], P=P)
print("\n getting index from the sampler...")
list_of_all_batches = list(sampler)

shuffled_indices = [index for batch in list_of_all_batches for index in batch]
print(f"Have created {len(shuffled_indices)} lines")

shuffled_df_train = trainingset.iloc[shuffled_indices]
shuffled_df_train = shuffled_df_train.reset_index(drop=True)

constructing PairedBatchSampler ...
Found 3827 unique (ref_id, filter) combinations
creating 'chunks' (size K=2)...
Already created 7729 'K-chunks'.

 getting index from the sampler...
Have created 15456 lines


In [9]:
#generate the stamp and mask
cutout_size = 48
shuffled_stamp_train = np.zeros((len(shuffled_df_train), cutout_size*2, cutout_size*2))
shuffled_mask_train = np.zeros((len(shuffled_df_train), cutout_size*2, cutout_size*2))

for i in tqdm(range(len(shuffled_df_train))):
    row = shuffled_df_train.iloc[[i]]
    path = shuffled_df_train['path'][i]
    image = fits.getdata(path)
    mask = fits.getdata(path.replace('.fits', '.mask.fits'))
    
    shuffled_stamp_train[i] = creat_stamps(image, row)
    shuffled_mask_train[i] = creat_stamps(mask, row)


shuffled_mask_gal = np.zeros((len(shuffled_df_train), cutout_size*2, cutout_size*2))
shuffled_mask_annulus = np.zeros((len(shuffled_df_train), cutout_size*2, cutout_size*2))
shuffled_area_annulus = np.zeros((len(shuffled_df_train)))

100%|██████████| 47552/47552 [1:16:39<00:00, 10.34it/s]


In [10]:
for i in tqdm(range(len(shuffled_df_train))):
    row = shuffled_df_train.iloc[[i]]
    aperture_x = cutout_size + row['aperture_x'].item() - math.floor(row['aperture_x'].item())
    aperture_y = cutout_size + row['aperture_y'].item() - math.floor(row['aperture_y'].item())

    shuffled_mask_gal[i] = gal_mask_generator(shuffled_mask_train[i], aperture_x, aperture_y,
                                   row['aperture_theta'].item(),row['aperture_a'].item(),row['aperture_b'].item())
    shuffled_mask_annulus[i], shuffled_area_annulus[i] = annulus_mask_generator(shuffled_mask_train[i], aperture_x, aperture_y)

100%|██████████| 47552/47552 [00:16<00:00, 2830.18it/s]


In [11]:
shuffled_mask_train = 1 - shuffled_mask_train

In [12]:
#To convert them into tensor
tensor_stamp_train = torch.from_numpy(shuffled_stamp_train).float()
tensor_mask_train = torch.from_numpy(shuffled_mask_train).float()
tensor_mask_gal = torch.from_numpy(shuffled_mask_gal).float()
tensor_mask_annulus = torch.from_numpy(shuffled_mask_annulus).float()
tensor_area_annulus = torch.from_numpy(shuffled_area_annulus).float()

tensor_stamp_train = tensor_stamp_train.unsqueeze(1)
tensor_mask_train = tensor_mask_train.unsqueeze(1)
tensor_mask_gal = tensor_mask_gal.unsqueeze(1)
tensor_mask_annulus = tensor_mask_annulus.unsqueeze(1)
tensor_area_annulus = tensor_area_annulus.unsqueeze(1)

#We just need 'ref_id', 'zp'
features_df_train = shuffled_df_train[['area', 'ref_id', 'zp']].values
features_df_train_tensor = torch.FloatTensor(features_df_train)

In [13]:
from torch.utils.data import DataLoader, TensorDataset

dataset = TensorDataset(tensor_stamp_train, tensor_mask_train, tensor_mask_gal, tensor_mask_annulus, tensor_area_annulus, features_df_train_tensor)
traindataloader = DataLoader(dataset, batch_size=16, shuffle=False)
#I set shuffle=False here, because I have already do the shuffle before

# Define the 2 new loss functios

In [14]:
#I have already simplified the loss functions

def loss_unbias(flux_gal_inputs, flux_gal_outputs):

    loss = (flux_gal_inputs - flux_gal_outputs).abs().sum()
    
    return loss
    
def PairedDifferenceLoss(flux_gal_calibrated_outputs):
    
    batch_size = flux_gal_calibrated_outputs.shape[0]
    half_B = batch_size // 2 #divide into 2
        
    outputs_1 = flux_gal_calibrated_outputs[0:half_B]
    outputs_2 = flux_gal_calibrated_outputs[half_B:]
    loss = (outputs_1 - outputs_2).abs().sum()
    
    return loss

# The construction of the LoRA-N2V model. 

In [15]:
#load the n2v model
import os
os.chdir('../pn2v/src/pn2v')
from core import prediction
from core import utils
from unet import UNet

device=utils.getDevice()

CUDA available? True


In [16]:
os.chdir('../../..')
os.chdir('./model saved')
model=torch.load("/N2V_used_in_LoRA.net")

/tmp/ipykernel_291/4130286196.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model=torch.load(path+"/best_conv_N2V_PAUdm.net")


In [17]:
for param in model.parameters():
    param.requires_grad = False

In [18]:
#print(model)
target_list = [
    'conv_final',
    'down_convs.0.conv1',
    'down_convs.0.conv2',
    'down_convs.1.conv1',
    'down_convs.1.conv2',
    'down_convs.2.conv1',
    'down_convs.2.conv2',
    'up_convs.0.conv1',
    'up_convs.0.conv2',
    'up_convs.1.conv1',
    'up_convs.1.conv2'
]

In [19]:
config = LoraConfig(
    r=8,  
    lora_alpha=16, 
    target_modules=target_list,
    lora_dropout=0.1,
)

lora_model = get_peft_model(model, config)
lora_model.print_trainable_parameters()

trainable params: 84,560 || all params: 1,761,938 || trainable%: 4.7993


# Train!

In [ ]:
num_epochs = 200
lora_model.train()
optimizer = torch.optim.AdamW(lora_model.parameters(), lr=1e-4)

for epoch in range(num_epochs):
    
    total_loss1 = 0.0
    total_loss2 = 0.0
    num_batches = 0
    
    progress_bar = tqdm(traindataloader, desc=f'Epoch {epoch+1}/{num_epochs}')
    
    for tensor_stamp_train, tensor_mask_train, tensor_mask_gal, tensor_mask_annulus, tensor_area_annulus, features_df_train_tensor in progress_bar:

        inputs = tensor_stamp_train.to(device)
        mask_abnormal = tensor_mask_train.to(device) 
        mask_gal = tensor_mask_gal.to(device)
        mask_annulus = tensor_mask_annulus.to(device)
        area_annulus = tensor_area_annulus.to(device)
        features = features_df_train_tensor.to(device)#'area', 'ref_id', 'zp'
        optimizer.zero_grad()
        
        outputs = lora_model(inputs) 
        #batch_size = inputs.shape[0]
        batch_size = inputs.shape[0]
        #get the flux
        flux_raw_inputs = torch.sum(inputs * mask_gal * mask_abnormal, dim=[2, 3])
        flux_raw_outputs = torch.sum(outputs * mask_gal * mask_abnormal, dim=[2, 3])
        flux_annulus_inputs = torch.sum(inputs * mask_annulus, dim=[2, 3])
        flux_annulus_outputs = torch.sum(outputs * mask_annulus, dim=[2, 3])
        flux_gal_inputs = flux_raw_inputs - flux_annulus_inputs * features[:,0].unsqueeze(1) / area_annulus
        flux_gal_outputs = flux_raw_outputs - flux_annulus_outputs * features[:,0].unsqueeze(1) / area_annulus
        #get the calibrated flux.
        #Note that here we just calculate the calibrated flux of the denoised image, we don't need to include the undenoised ones
        flux_gal_calibrated_outputs = flux_gal_outputs * features[:,2].unsqueeze(1)
        
        loss1 = loss_unbias(flux_gal_inputs, flux_gal_outputs) / batch_size
        loss2 = PairedDifferenceLoss(flux_gal_calibrated_outputs) / batch_size#(batch_size/2)
        loss = loss1 + loss2
        loss.backward()
        optimizer.step()
        
        current_loss1 = loss1.item()
        current_loss2 = loss2.item()
        total_loss1 += current_loss1
        total_loss2 += current_loss2
        num_batches += 1
        progress_bar.set_postfix({
            'Loss of this batch': f'{current_loss1 + current_loss2:.6f}',
            'Avg Loss': f'{(current_loss1+current_loss2)/num_batches:.6f}'
        })
    
    avg_loss1 = total_loss1 / num_batches
    avg_loss2 = total_loss2 / num_batches
    
    print(f"\nEpoch {epoch+1} Completed - Average unbias Loss: {avg_loss1:.6f} Average df Loss {avg_loss2:.6f}")

Epoch 1/200: 100%|██████████| 2972/2972 [00:40<00:00, 73.56it/s, Loss of this batch=3.912856, Avg Loss=0.001317]  



Epoch 1 Completed - Average unbias Loss: 4.093553 Average df Loss 2.736902


Epoch 2/200: 100%|██████████| 2972/2972 [00:35<00:00, 84.21it/s, Loss of this batch=3.888089, Avg Loss=0.001308]  



Epoch 2 Completed - Average unbias Loss: 3.610890 Average df Loss 2.959443


Epoch 3/200: 100%|██████████| 2972/2972 [00:34<00:00, 85.10it/s, Loss of this batch=3.683333, Avg Loss=0.001239]  


...

Epoch 198/200: 100%|██████████| 2972/2972 [00:35<00:00, 84.23it/s, Loss of this batch=4.273563, Avg Loss=0.001438]  



Epoch 198 Completed - Average unbias Loss: 2.421372 Average df Loss 3.034972


Epoch 199/200: 100%|██████████| 2972/2972 [00:35<00:00, 84.39it/s, Loss of this batch=4.384925, Avg Loss=0.001475]  



Epoch 199 Completed - Average unbias Loss: 2.453131 Average df Loss 3.022395


Epoch 200/200:   8%|▊         | 252/2972 [00:03<00:33, 82.37it/s, Loss of this batch=4.464943, Avg Loss=0.017441] 

In [ ]:
final_model_path = "./name_the_model.pth"
torch.save(lora_model, final_model_path)